# Welcome to your Lakehouse Lab workspace

Everything here runs **as you**: Trino, the Iceberg catalog, DuckDB, dbt and Spark all use your own login token. Nobody pastes passwords or storage keys into a notebook.

| Tool | How to reach it |
|---|---|
| Trino (SQL warehouse) | `lakehouse.trino_connection()`, or `%sql` with `lakehouse.sql_engine()` |
| Iceberg catalog (PyIceberg) | `lakehouse.catalog()` |
| DuckDB (single-node SQL) | `lakehouse.duckdb_connect()` |
| dbt | `dbt build` in `~/starter/dbt_lakehouse` |
| Spark (profile `engineer`) | `lakehouse.spark()` |
| Your token, for anything else | `lakehouse.lab_token()` or `lab-token` in a terminal |

The sample data lives in `lakehouse.samples`: the TPC-H tables `region`, `nation`, `customer`, `orders` and `lineitem`. This folder is yours: changes you make are kept, and the lab never overwrites it.

## 1. Who am I?

`lab_token()` gets your current access token from JupyterHub, which renews it for you while the workspace runs. Every helper below uses it; you rarely need to call it yourself.

In [ ]:
import lakehouse

claims = lakehouse.token_claims(lakehouse.lab_token())
print("user:  ", claims["preferred_username"])
print("groups:", claims.get("groups"))

## 2. Query Trino

Trino is the shared SQL engine. What you may read or write is decided by your lab group (`lab-admin`, `engineer`, `analyst` or `viewer`).

In [ ]:
cur = lakehouse.trino_connection().cursor()
cur.execute("""
    SELECT r.name AS region, count(*) AS customers
    FROM lakehouse.samples.customer c
    JOIN lakehouse.samples.nation n ON c.nationkey = n.nationkey
    JOIN lakehouse.samples.region r ON n.regionkey = r.regionkey
    GROUP BY r.name ORDER BY customers DESC
""")
cur.fetchall()

### SQL cells

JupySQL gives you `%%sql` cells against the same Trino connection.

In [ ]:
%load_ext sql
trino = lakehouse.sql_engine()
%sql trino

In [ ]:
%%sql
SELECT orderpriority, count(*) AS orders, round(sum(totalprice), 2) AS revenue
FROM lakehouse.samples.orders
GROUP BY orderpriority
ORDER BY orderpriority

## 3. DuckDB on the same tables

DuckDB attaches the Iceberg catalog directly. The catalog checks your permissions and hands DuckDB short-lived storage credentials for just the tables you read. For a connection you keep open for hours, call `lakehouse.attach_lakehouse(con)` again to renew the token.

In [ ]:
con = lakehouse.duckdb_connect()
con.sql("""
    SELECT o.orderstatus, count(*) AS orders, round(sum(l.extendedprice * (1 - l.discount)), 2) AS net
    FROM lakehouse.samples.orders o
    JOIN lakehouse.samples.lineitem l ON o.orderkey = l.orderkey
    GROUP BY 1 ORDER BY 1
""")

## 4. The catalog from Python (PyIceberg)

`lakehouse.catalog()` is the same as `pyiceberg.catalog.load_catalog("lakehouse")`.

In [ ]:
cat = lakehouse.catalog()
print(cat.list_tables("samples"))
nation = cat.load_table(("samples", "nation"))
nation.scan(limit=5).to_pandas()

## 5. dbt

`~/starter/dbt_lakehouse` is a small dbt project over the samples: staging views, then `dim_customers`, `fct_orders` and `revenue_by_region`. It writes to your own schema, `lakehouse.dbt_<your user name>`, so it needs write access (groups `engineer` or `lab-admin`).

Open a terminal (or VS Code from the launcher) and run:

```bash
cd ~/starter/dbt_lakehouse
dbt build
```

The `dbt` command hands dbt your token automatically. The same works from this notebook:

In [ ]:
!cd ~/starter/dbt_lakehouse && dbt build

## 6. Spark (profile `engineer` only)

The lab runs one shared Spark Connect server. `lakehouse.spark()` opens your own session on it. Use a plain `DROP TABLE` in Spark, never `DROP TABLE ... PURGE`: the catalog cleans up the files itself.

In [ ]:
spark = lakehouse.spark()
spark.sql("SELECT count(*) FROM lakehouse.samples.lineitem").show()